In [1]:
%pip install -U ultralytics pyyaml matplotlib

     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     ---------------------------------------- 52.8/52.8 kB 1.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 39.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/158.8 kB ? eta -:--:--
   ---------------------------------------- 158.8/158.8 kB ? eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.1 MB 27.2 MB/s eta 0:00:01
   ----------- ---------------------------- 2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires PyYAML==6.0.1, but you have pyyaml 6.0.3 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
!pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------ --------------------------- 3.1/9.9 MB 20.5 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 29.4 MB/s  0:00:00


In [1]:
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO

In [23]:
%pip install -U ipywidgets

Note: you may need to restart the kernel to use updated packages.


GPU 확인

In [2]:
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU를 못 잡았습니다. CUDA / torch 설치 확인하십쇼.")

torch version: 2.7.1+cu118
CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 4070


기본 설정

In [3]:
# 결과 저장 폴더
PROJECT_ROOT = Path.cwd() / "runs_ppe"
PROJECT_ROOT.mkdir(exist_ok=True)

# 4070 기준
DEVICE = 0
WORKERS = 8
IMGSZ = 640

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\leej1\OneDrive\바탕 화면\DATA\runs_ppe


1차 학습용 기본 모델 선택

In [4]:
# ultralytics 버전에 따라 yolo26n.pt 가 안 될 수도 있어서 fallback 넣음
CANDIDATE_MODELS = ["yolo26n.pt", "yolo11n.pt", "yolov8n.pt"]

selected_model_name = None
model = None

for m in CANDIDATE_MODELS:
    try:
        model = YOLO(m)
        selected_model_name = m
        print(f"사용할 모델: {m}")
        break
    except Exception as e:
        print(f"{m} 로드 실패: {e}")

if model is None:
    raise RuntimeError("기본 모델 로드 실패. ultralytics 버전 확인 필요.")

사용할 모델: yolo26n.pt


1차 학습 설정

In [5]:
STAGE1_EPOCHS = 30
STAGE1_BATCH = -1   # GPU 메모리에 맞춰 자동 조절
STAGE1_DATA = "construction-ppe.yaml"

print("Stage 1 data:", STAGE1_DATA)
print("Stage 1 epochs:", STAGE1_EPOCHS)
print("Stage 1 batch:", STAGE1_BATCH)
print("Stage 1 model:", selected_model_name)

Stage 1 data: construction-ppe.yaml
Stage 1 epochs: 30
Stage 1 batch: -1
Stage 1 model: yolo26n.pt


1차 학습 실행

In [6]:
stage1_results = model.train(
    data=STAGE1_DATA,
    epochs=STAGE1_EPOCHS,
    imgsz=IMGSZ,
    batch=STAGE1_BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_ROOT),
    name="stage1_construction_ppe",
    pretrained=True,
    cache=False,
    amp=True,
    patience=20,
    plots=True
)

Ultralytics 8.4.34  Python-3.11.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=construction-ppe.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=stage1_construction_ppe, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

헬멧만 확인

In [68]:
import pandas as pd

# 1차 학습 모델 검증 (기본 전체 표 출력 끄기)
stage1_metrics = model.val(
    data=STAGE1_DATA,
    imgsz=IMGSZ,
    batch=16,
    device=DEVICE,
    plots=True,
    verbose=False
)

# 클래스 이름 가져오기
names = stage1_metrics.names if hasattr(stage1_metrics, "names") else model.names

# 보고 싶은 헬멧 관련 클래스만 지정
target_classes = {"helmet", "no_helmet", "helmet_on", "helmet_off"}

rows = []
for cls_idx, cls_name in names.items():
    if cls_name in target_classes:
        p, r, map50, map5095 = stage1_metrics.box.class_result(cls_idx)
        rows.append({
            "Class": cls_name,
            "P": round(p, 4),
            "R": round(r, 4),
            "mAP50": round(map50, 4),
            "mAP50-95": round(map5095, 4)
        })

if rows:
    helmet_df = pd.DataFrame(rows)
    display(helmet_df)
else:
    print("헬멧 관련 클래스가 없습니다.")
    print("현재 클래스 목록:", names)

Ultralytics 8.4.34  Python-3.11.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 1030.0616.1 MB/s, size: 82.2 KB)
val: Scanning C:\Users\leej1\datasets\construction-ppe\labels\val.cache... 143 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 143/143  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 5.1it/s 1.8s0.1ss
                   all        143       1172      0.554      0.422      0.426      0.205
Speed: 2.4ms preprocess, 5.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\leej1\OneDrive\ \DATA\runs\detect\val9


,Class,P,R,mAP50,mAP50-95
0,helmet,0.5633,0.8259,0.7654,0.4178
1,no_helmet,0.3909,0.1429,0.2045,0.0608


1차 학습 모델 검증

In [7]:
stage1_metrics = model.val(
    data=STAGE1_DATA,
    imgsz=IMGSZ,
    batch=16,
    device=DEVICE,
    plots=True
)

stage1_metrics

Ultralytics 8.4.34  Python-3.11.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO26n summary (fused): 122 layers, 2,376,981 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1131.1716.6 MB/s, size: 102.9 KB)
val: Scanning C:\Users\leej1\datasets\construction-ppe\labels\val.cache... 143 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 143/143  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 7.2it/s 1.2s<0.1s
                   all        143       1172      0.645      0.473      0.491       0.25
                helmet        107        201      0.609      0.796      0.779      0.426
                gloves         68        136      0.587      0.713      0.702      0.336
                  vest        109        171      0.643      0.795      0.763      0.472
                 boots         64        151      0.586      0.735      0.736      0.411
               goggl

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000018983F12710>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,  

1차 학습 이미지 예측 테스트

In [15]:
from pathlib import Path
import shutil

# 1차 학습이 끝난 뒤 저장된 best.pt 경로
best_model_path = Path(model.trainer.best)
last_model_path = Path(model.trainer.last)

print("1차 학습 best 모델:", best_model_path)
print("1차 학습 last 모델:", last_model_path)

# 형님이 따로 보관할 폴더
SAVE_DIR = Path(r"C:\ppe_models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 원하는 이름으로 복사 저장
saved_best_path = SAVE_DIR / "stage1_best.pt"
saved_last_path = SAVE_DIR / "stage1_last.pt"

shutil.copy2(best_model_path, saved_best_path)
shutil.copy2(last_model_path, saved_last_path)

print("복사 저장 완료")
print("best 저장 위치:", saved_best_path)
print("last 저장 위치:", saved_last_path)

1차 학습 best 모델: C:\Users\leej1\OneDrive\바탕 화면\DATA\runs_ppe\stage1_construction_ppe\weights\best.pt
1차 학습 last 모델: C:\Users\leej1\OneDrive\바탕 화면\DATA\runs_ppe\stage1_construction_ppe\weights\last.pt
복사 저장 완료
best 저장 위치: C:\ppe_models\stage1_best.pt
last 저장 위치: C:\ppe_models\stage1_last.pt
